# Microscope calibration and droplet spacing (n26)

Five microscope pictures of the droplet stream below the nozzle, taken with the microscope fixed while the manipulator moved. The file names give the manipulator position to 0.01 mm (`xm2p47mm_y8p5mm_z0p71mm` = x -2.47, y 8.5, z 0.71 mm).

1. Find the nozzle's lower edge and the jet in each image, and how far the image moved relative to y = 9.0 mm.
2. Scale (µm/px) from the y scan (8.5, 8.9, 9.0 mm): y runs along the jet, in the image plane.
3. Cross-check with the x and z steps.
4. Droplet spacing: period of the droplet train along the jet, times the scale.
5. Jet direction: the part of the jet's tilt the microscope can see, as a condition on x and z.
6. Both components of the jet direction, adding the beam's x scans.

In [ ]:
%matplotlib inline
%config InlineBackend.figure_format='retina'
import re

from veritas_paths import REPO_ROOT

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.ndimage import gaussian_filter, gaussian_filter1d, map_coordinates
from scipy.optimize import curve_fit
from scipy.signal import find_peaks
from scipy.special import erf

import daq

In [ ]:
IMG_DIR = REPO_ROOT / "analysis_notebooks" / "iason"
POS_ERR = 0.010  # mm, accuracy of the manipulator positions in the file names
REF = (-2.47, 9.0, 0.71)  # (x, y, z) mm of the reference image, the one with the nozzle highest in the frame


def position(path):
    """(x, y, z) in mm from a name like '..._xm2p47mm_y8p5mm_z0p71mm.png' (m = minus, p = decimal point)."""
    found = dict(re.findall(r"_([xyz])(m?\d+(?:p\d+)?)mm", path.stem))
    return tuple(float(found[a].replace("m", "-").replace("p", ".")) for a in "xyz")


# green channel only: the backlight is green, red and blue carry little signal
paths = sorted(IMG_DIR.glob("veritas_nov26_droplets_*.png"))
images = {position(p): plt.imread(p)[..., 1].astype(float) for p in paths}
Y_SCAN = sorted((p for p in images if (p[0], p[2]) == (REF[0], REF[2])), key=lambda p: -p[1])
print(len(images), "images of", images[REF].shape, "px; y scan:", [p[1] for p in Y_SCAN], "mm")

In [ ]:
fig, axs = plt.subplots(2, 3, figsize=(12, 6.5))
for ax, (pos, img) in zip(axs.flat, sorted(images.items())):
    ax.imshow(img, cmap="gray")
    ax.set_title("x = %.2f, y = %.2f, z = %.2f mm" % pos, fontsize=9)
for ax in axs.flat:
    ax.axis("off")
fig.tight_layout()

## 1. Nozzle edge, jet, and image shifts

The nozzle's lower edge (bottom of the dark block) is a dark-to-bright step: an error-function fit in each 8-px column band gives its row. The jet is a faint dark line: a Gaussian dip fitted in 100-row blocks gives its column, and a straight line through those its track.

Shifts relative to `REF`:
- `dy_px`: median over columns of the edge-row difference, leaving out the columns where the jet stub hangs below the edge. `edge_scatter_px` is the spread around it: about 1 px for the y scan means the nozzle moved rigidly. The x and z steps also move the nozzle sideways and in depth, so their edges do not overlay as well (4-5 px); their dy is not used.
- `dx_px`: change in the jet's column at the same height below the nozzle.
- `dy_droplets_px`: the shift that best overlays the droplet train on REF's. The strobe is locked to the droplet generator, so the droplets sit at fixed places below the nozzle and move with it. This measures the shift at the jet's depth; the blurred nozzle edge may be at a different depth.

In [ ]:
EDGE_COLS = np.arange(640, 2112, 8)  # columns along the nozzle's lower edge, clear of its corners
JET_COLS = slice(1100, 1500)  # the jet stays inside these columns in all images
JET_ROW0 = 700  # first row used for the jet track, below the nozzle in all images


def _step(r, r0, w, lo, hi):
    return lo + (hi - lo) * 0.5 * (1 + erf((r - r0) / (np.sqrt(2) * w)))


def _dip(c, mu, sigma, depth, base):
    return base - depth * np.exp(-0.5 * ((c - mu) / sigma) ** 2)


def nozzle_edge(img, band=4, half=60):
    """Row of the nozzle's lower edge at each column in EDGE_COLS, from an erf fit (NaN where it fails)."""
    rows = np.full(len(EDGE_COLS), np.nan)
    for i, c in enumerate(EDGE_COLS):
        p = img[:, c - band : c + band].mean(1)
        r = int(np.argmax(np.gradient(gaussian_filter1d(p, 4))[:1000]))  # steepest dark -> bright step
        sl = slice(max(r - half, 0), r + half)
        try:
            p0 = [r, 10, p[sl][:10].mean(), p[sl][-10:].mean()]
            rows[i] = curve_fit(_step, np.arange(len(p))[sl], p[sl], p0=p0)[0][0]
        except RuntimeError:
            pass
    return rows


def jet_line(img):
    """(slope, intercept) of the jet's column against row."""
    hp = img - gaussian_filter(img, 25)  # drop the illumination gradient
    rows, cols = [], []
    for r0 in range(JET_ROW0, img.shape[0] - 100, 100):
        q = gaussian_filter1d(hp[r0 : r0 + 100].mean(0), 2)
        c0 = np.argmin(q[JET_COLS]) + JET_COLS.start
        cc = np.arange(c0 - 20, c0 + 21)
        cols.append(curve_fit(_dip, cc, q[cc], p0=[c0, 4, -q[c0], 0])[0][0])
        rows.append(r0 + 50)
    return np.polyfit(rows, cols, 1)


def jet_profile(img, line):
    """Droplet shadow along the jet, per row: local background minus an 11-px band on the jet."""
    rows = np.arange(img.shape[0])
    centre = np.polyval(line, rows)

    def band(offsets):
        cols = np.clip(np.round(centre[:, None] + offsets).astype(int), 0, img.shape[1] - 1)
        return img[rows[:, None], cols].mean(1)

    return band(np.r_[-30:-15, 16:31]) - band(np.arange(-5, 6))


lines = {p: jet_line(img) for p, img in images.items()}
profiles = {p: jet_profile(img, lines[p]) for p, img in images.items()}
edges = {p: nozzle_edge(img) for p, img in images.items()}
for p, e in edges.items():
    e[np.abs(EDGE_COLS - np.polyval(lines[p], 300)) < 40] = np.nan  # the jet stub hangs below the edge here


def edge_at_jet(pos):
    """Row of the nozzle edge where the jet leaves it, interpolated across the masked columns."""
    ok = np.isfinite(edges[pos])
    return np.interp(np.polyval(lines[pos], 300), EDGE_COLS[ok], edges[pos][ok])

In [ ]:
DROP_ROWS = np.arange(230, 1450)  # REF rows from just below the nozzle to where the y = 8.5 mm image ends


def droplet_dy(pos, dy_guess):
    """Shift (px) that best overlays the droplet train of `pos` on REF's, searched within ±50 px of dy_guess."""
    ref, p = gaussian_filter1d(profiles[REF], 3), gaussian_filter1d(profiles[pos], 3)

    def corr(d):
        return np.corrcoef(ref[DROP_ROWS], np.interp(DROP_ROWS + d, np.arange(len(p)), p))[0, 1]

    coarse = np.arange(round(dy_guess) - 50, round(dy_guess) + 51)  # under half a droplet period
    best = coarse[np.argmax([corr(d) for d in coarse])]
    fine = np.arange(best - 1, best + 1.001, 0.05)
    return fine[np.argmax([corr(d) for d in fine])]


rows = []
for pos in images:
    diff = edges[pos] - edges[REF]
    dy = np.nanmedian(diff)
    dx = np.polyval(lines[pos], 800 + dy) - np.polyval(lines[REF], 800)  # same height below the nozzle
    rows.append(dict(x=pos[0], y=pos[1], z=pos[2], dx_px=dx, dy_px=dy, edge_scatter_px=np.nanstd(diff - dy),
                     dy_droplets_px=droplet_dy(pos, dy)))
shifts = pd.DataFrame(rows).set_index(["x", "y", "z"]).sort_index()
shifts.round(2)

In [ ]:
# what the fits see: nozzle edge (red), jet track (cyan), REF's edge (dashed)
fig, axs = plt.subplots(1, 3, figsize=(13, 4.5))
ref_edge = np.nanmedian(edges[REF])
for ax, pos in zip(axs, Y_SCAN):
    img = images[pos]
    ax.imshow(img, cmap="gray")
    ax.plot(EDGE_COLS, edges[pos], "r-", lw=1)
    r = np.arange(JET_ROW0, img.shape[0])
    ax.plot(np.polyval(lines[pos], r), r, "c-", lw=0.6)
    ax.axhline(ref_edge, color="w", ls="--", lw=0.8)
    ax.set_xlim(400, 2300)
    ax.set_title(f"y = {pos[1]} mm: dy = {shifts.loc[pos].dy_px:.1f} px", fontsize=9)
    ax.axis("off")
fig.tight_layout()

## 2. Scale from the y scan

Weighted straight-line fit of manipulator y against image travel, with ±10 µm on each y. The image travel is the length of (dx, dy): the y axis's image is rolled about 1.4° from the pixel columns, which changes the length by under 0.1 %. The ±10 µm on the positions sets the error; the pixel positions are good to ~0.2 px (0.3 µm).

In [ ]:
y_mm = np.array([p[1] for p in Y_SCAN])
scan = shifts.loc[Y_SCAN]


def fit_scale(dx, dy):
    """µm/px, its error, fit coefficients and chi2 from y = y0 - scale * travel, travel = |(dx, dy)| signed by dy."""
    travel = np.sign(dy) * np.hypot(dx, dy)
    coef, cov = np.polyfit(travel, y_mm, 1, w=np.full(len(y_mm), 1 / POS_ERR), cov="unscaled")
    chi2 = np.sum(((y_mm - np.polyval(coef, travel)) / POS_ERR) ** 2)
    return -1e3 * coef[0], 1e3 * np.sqrt(cov[0, 0]), coef, travel, chi2


SCALE, SCALE_ERR, coef, travel, chi2 = fit_scale(scan.dx_px, scan.dy_px)
s_drop, s_drop_err, *_ = fit_scale(scan.dx_px, scan.dy_droplets_px)
print(f"nozzle edge: {SCALE:.4f} ± {SCALE_ERR:.4f} µm/px  ({1e3 / SCALE:.1f} px/mm), "
      f"chi2 = {chi2:.3f} for {len(y_mm) - 2} dof")
print(f"droplets:    {s_drop:.4f} ± {s_drop_err:.4f} µm/px  ({100 * (s_drop / SCALE - 1):+.2f} % from the edge)")
print(f"field of view: {images[REF].shape[1] * SCALE / 1e3:.2f} x {images[REF].shape[0] * SCALE / 1e3:.2f} mm")

fig, (a, b) = plt.subplots(1, 2, figsize=(10, 3.5))
a.errorbar(travel, y_mm, yerr=POS_ERR, fmt="o")
t = np.linspace(-20, travel.max() + 20, 2)
a.plot(t, np.polyval(coef, t), "-", label=f"{SCALE:.3f} µm/px")
a.set(xlabel="image travel from y = 9.0 mm (px)", ylabel="manipulator y (mm)")
a.legend()
b.errorbar(y_mm, 1e3 * (y_mm - np.polyval(coef, travel)), yerr=1e3 * POS_ERR, fmt="o")
b.axhline(0, color="k", lw=0.5)
b.set(xlabel="manipulator y (mm)", ylabel="y - fit (µm)")
fig.tight_layout()

## 3. Cross-check with the x and z steps

Steps of +0.1 mm in x and in z both move the image sideways (x to the right, z to the left), by about 0.7 of the step. With x and z horizontal and the microscope looking horizontally at an angle φ from z, the projections are sin φ and cos φ, so their squares should add to 1. Each step is the difference of two positions known to ±10 µm, so a projection carries about ±0.14 from the positions alone: this only checks the scale is not far off.

In [ ]:
C_COL = {}  # image columns per mm of manipulator move, from the single 0.1 mm steps
for axis, i in [("x", 0), ("z", 2)]:
    pos = next(p for p in images if p[i] != REF[i] and p[1] == REF[1])
    C_COL[axis] = shifts.loc[pos].dx_px / (pos[i] - REF[i])
    print(f"{axis}: {1e3 * (pos[i] - REF[i]):+.0f} µm step -> image moves {shifts.loc[pos].dx_px * SCALE:+.1f} µm "
          f"sideways, projection {abs(C_COL[axis]) * SCALE / 1e3:.2f}")
P_X, P_Z = C_COL["x"] * SCALE / 1e3, C_COL["z"] * SCALE / 1e3  # µm sideways in the image per µm of move
print(f"sum of squares = {P_X ** 2 + P_Z ** 2:.2f} (expect 1); "
      f"microscope axis {np.degrees(np.arctan2(abs(P_X), abs(P_Z))):.0f} deg from z")

## 4. Droplet spacing

The profile along the jet is periodic. Its period comes from the FFT peak, refined by fitting three harmonics with the period free. `period_px` uses the whole train from 150 px below the nozzle edge, skipping the jet stub and the break-up region.

The same fit in a three-droplet window sliding down the jet shows the spacing is 2-3 % longer in the first ~0.8 mm and then levels off. The y = 8.5 mm image sits 362 px lower in the frame: against distance from the nozzle it overlays the others, against image row it does not. So the change is most likely in the jet (droplets slowing after break-up), not lens distortion. `settled_px` fits only from 600 px (0.83 mm) below the edge. The scatter of the five images sets the period error.

In [ ]:
N_HARM = 3
START_BELOW_EDGE = 150  # px, skips the jet stub and the break-up region
SETTLED_BELOW_EDGE = 600  # px, where the local spacing has levelled off
WINDOW = 333  # px, three droplets, for the local spacing


def harmonic(r, period, c, *ab):
    w = 2 * np.pi * r / period
    return c + sum(a * np.cos(k * w) + b * np.sin(k * w) for k, (a, b) in enumerate(zip(ab[::2], ab[1::2]), 1))


def droplet_period(profile, start, stop):
    """Harmonic-fit parameters (period first, in px) for profile[start:stop]."""
    r = np.arange(start, stop)
    y = profile[r]
    f = np.fft.rfftfreq(2**16)
    spec = np.abs(np.fft.rfft(y - y.mean(), 2**16))
    ok = (f > 1 / 200) & (f < 1 / 60)
    p0 = [1 / f[ok][np.argmax(spec[ok])], y.mean(), 1, 1] + [0, 0] * (N_HARM - 1)
    return curve_fit(harmonic, r, y, p0=p0)[0]


rows, fits = [], {}
for pos, prof in sorted(profiles.items()):
    edge = edge_at_jet(pos)
    start, stop = int(edge) + START_BELOW_EDGE, len(prof) - 10
    fits[pos] = (edge, start, stop, droplet_period(prof, start, stop))
    rows.append(dict(x=pos[0], y=pos[1], z=pos[2], droplets=(stop - start) / fits[pos][3][0],
                     period_px=fits[pos][3][0],
                     settled_px=droplet_period(prof, int(edge) + SETTLED_BELOW_EDGE, stop)[0]))
periods = pd.DataFrame(rows).set_index(["x", "y", "z"])
periods[["period_um", "settled_um"]] = periods[["period_px", "settled_px"]].to_numpy() * SCALE
periods.round(2)

In [ ]:
# local spacing: the same fit in a three-droplet window sliding down the jet,
# against image row (left) and against distance below the nozzle (right)
fig, (a, b) = plt.subplots(1, 2, figsize=(13, 3.8), sharey=True)
for pos, (edge, start, stop, _) in fits.items():
    centres = np.arange(start, stop - WINDOW, 28) + WINDOW / 2
    local = SCALE * np.array([droplet_period(profiles[pos], int(c - WINDOW / 2), int(c + WINDOW / 2))[0]
                              for c in centres])
    label = "y=%.1f x=%.2f z=%.2f" % (pos[1], pos[0], pos[2])
    a.plot(centres, local, ".-", ms=3, label=label)
    b.plot((centres - edge) * SCALE, local, ".-", ms=3)
b.axvline(SETTLED_BELOW_EDGE * SCALE, color="k", lw=0.5, ls="--")
a.set(xlabel="window centre, image row (px)", ylabel="local droplet spacing (µm)")
b.set(xlabel="window centre below the nozzle edge (µm)")
a.legend(fontsize=7)
fig.tight_layout()

In [ ]:
def spacing(col):
    """Mean period (px) over the images and its error, and the spacing in µm with the scale error added."""
    P, P_err = periods[col].mean(), periods[col].std(ddof=1) / np.sqrt(len(periods))
    return P, P_err, P * SCALE, P * SCALE * np.hypot(P_err / P, SCALE_ERR / SCALE)


print(f"scale {SCALE:.4f} ± {SCALE_ERR:.4f} µm/px")
for col, start in [("period_px", START_BELOW_EDGE), ("settled_px", SETTLED_BELOW_EDGE)]:
    P, P_err, s, s_err = spacing(col)
    print(f"from {start * SCALE:4.0f} µm below the edge: period {P:.2f} ± {P_err:.2f} px "
          f"-> droplet spacing {s:.1f} ± {s_err:.1f} µm")

# profiles against distance below the nozzle edge, offset per image; the dotted lines are one spacing apart
fig, ax = plt.subplots(figsize=(12, 5))
for k, (pos, (edge, start, stop, popt)) in enumerate(fits.items()):
    d_um = (np.arange(len(profiles[pos])) - edge) * SCALE
    off = 30 * k / 255  # plt.imread scales to 0..1
    ax.plot(d_um, gaussian_filter1d(profiles[pos], 3) + off, color="0.6", lw=0.7)
    r = np.arange(start, stop)
    ax.plot(d_um[r], harmonic(r, *popt) + off, lw=1)
    ax.text(-40, off, "y=%.1f x=%.2f z=%.2f" % (pos[1], pos[0], pos[2]), ha="right", va="bottom", fontsize=8)
for m in np.arange((fits[REF][1] - fits[REF][0]) * SCALE, 2500, spacing("period_px")[2]):
    ax.axvline(m, color="k", lw=0.4, ls=":")
ax.set(xlabel="distance below the nozzle edge (µm)", ylabel="droplet shadow (offset)", xlim=(-350, 2300))
ax.set_yticks([])
fig.tight_layout()

### Droplet width across the jet

Each droplet shows as a dash, smeared along the jet during the light pulse; across the jet it is not smeared. The profile across the dash cores (±15 rows around each droplet), sampled on the jet line, gives the shadow's FWHM. As a diameter it is good to a few µm only. A moving sphere's shadow has FWHM ≈ 0.87 d with perfect optics, blur widens it, and the droplets are transparent: the bright fringe on one side is refracted light, which also makes the shadow shallow.

In [ ]:
OFF = np.arange(-40, 40.01, 0.5)  # px across the jet
CORE = 15  # rows either side of a droplet's centre


def across(img, line, rows):
    """Mean profile across the jet over `rows`, sampled at OFF px from the jet line."""
    cols = np.polyval(line, rows)[:, None] + OFF
    return map_coordinates(img, [np.broadcast_to(rows[:, None], cols.shape), cols], order=1).mean(0)


def _dip_lin(x, mu, sigma, depth, b0, b1):
    return b0 + b1 * x - depth * np.exp(-0.5 * ((x - mu) / sigma) ** 2)


def width_fit(prof):
    """Gaussian-dip parameters (mu, sigma, depth, b0, b1) of a profile across the jet."""
    return curve_fit(_dip_lin, OFF, prof, p0=[0, 4, np.ptp(prof) / 2, prof.mean(), 0])[0]


dash, gap = [], []
for pos, (edge, start, stop, popt) in fits.items():
    P = popt[0]
    centres = find_peaks(gaussian_filter1d(profiles[pos], 10)[start:stop], distance=int(0.6 * P))[0] + start
    for c in centres[(centres - CORE > start) & (centres + P / 2 + 6 < len(profiles[pos]))]:
        dash.append(across(images[pos], lines[pos], np.arange(c - CORE, c + CORE + 1)))
        gap.append(across(images[pos], lines[pos], np.arange(int(c + P / 2) - 5, int(c + P / 2) + 6)))
dash, gap = np.array(dash), np.array(gap)

p = width_fit(dash.mean(0))
per_drop = []
for prof in dash:
    try:
        per_drop.append(2.355 * abs(width_fit(prof)[1]) * SCALE)
    except RuntimeError:
        pass
print(f"{len(dash)} droplets: mean profile FWHM {2.355 * abs(p[1]):.1f} px = {2.355 * abs(p[1]) * SCALE:.1f} µm, "
      f"{100 * p[2] / p[3]:.0f} % deep")
print(f"one droplet at a time: median FWHM {np.median(per_drop):.1f} µm, "
      f"quartiles {np.percentile(per_drop, 25):.1f}-{np.percentile(per_drop, 75):.1f} µm")
print(f"between droplets: {100 * (1 - gap.mean(0)[np.abs(OFF) < 4].mean() / gap.mean(0)[np.abs(OFF) > 22].mean()):.1f} % deep")

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(OFF * SCALE, dash.mean(0), label="through the droplets")
ax.plot(OFF * SCALE, gap.mean(0), label="between droplets")
ax.plot(OFF * SCALE, _dip_lin(OFF, *p), "k:", lw=0.8, label="Gaussian fit")
ax.set(xlabel="across the jet (µm)", ylabel="intensity")
ax.legend(fontsize=8)
fig.tight_layout()

Notes
- The scale error is set by the ±10 µm on the manipulator positions. The three y points agree far better than that (chi2 ≪ 1), so the true error is probably smaller.
- The droplet train gives the same scale as the nozzle edge, so the magnification is the same at the jet's depth.
- The spacing is v / f, for droplet speed v and drop rate f, so the piezo frequency gives the droplet speed. The settled spacing is the one that applies further down the jet.

## 5. Jet direction in the microscope's view

The microscope looks along a horizontal line about 45° between x and z (section 3). A single view therefore shows the jet's tilt only along the picture's horizontal axis, which is one mix of x and z; tilt along the viewing axis is invisible.

Write the jet's sideways drift per mm downstream as a = dx/dh and c = dz/dh in manipulator axes. A 1 mm move of axis i shifts the picture by C_i columns and R_i rows. The jet's slope in the picture (columns per row) is then k = (C_x a + C_z c − C_y) / (−R_y); the R_x a and R_z c terms change it by under 0.1 %. The slopes of the five images therefore fix T = p_x a + p_z c, where p_i = C_i × scale is the image travel per µm of move.

To keep the jet in the same place in the picture while y changes, x and z must follow p_x dx/dy + p_z dz/dy = −T. Any (dx/dy, dz/dy) on that line keeps it in place; section 6 adds the beam's x scans to pick the point on it.

In [ ]:
# y entries from the y scan: columns and rows per mm of y
(C_Y, _), cov = np.polyfit(y_mm, scan.dx_px, 1, cov=True)
C_Y_ERR = np.sqrt(cov[0, 0])
R_Y = np.polyfit(y_mm, scan.dy_px, 1)[0]

k = np.array([lines[p][0] for p in images])  # jet slope in each picture, columns per row
K, K_ERR = k.mean(), k.std(ddof=1) / np.sqrt(len(k))

T = (C_Y - R_Y * K) * SCALE / 1e3
T_ERR = np.hypot(C_Y_ERR, R_Y * K_ERR) * SCALE / 1e3
lean = np.degrees(np.arctan(K) - np.arctan(C_Y / R_Y))
# the same from the y scan alone: the jet's column at a fixed row of the picture, against y
at_row = np.polyfit(y_mm, [np.polyval(lines[p], 1200) for p in Y_SCAN], 1)[0] * SCALE / 1e3

print(f"picture: y axis slope {C_Y / R_Y:.4f}, jet slope {K:.4f} ± {K_ERR:.4f} -> the jet leans {lean:+.2f} deg "
      "from the y axis (+ = towards +columns)")
print(f"T = p_x dx/dh + p_z dz/dh = {T:+.4f} ± {T_ERR:.4f}  (from the jet at a fixed row: {at_row:+.4f}); "
      f"p_x = {P_X:+.3f}, p_z = {P_Z:+.3f}")
print(f"to keep the jet in place per mm of y: {P_X:+.3f} dx/dy {P_Z:+.3f} dz/dy = {-T:+.4f}, e.g. "
      f"dx/dy = {-T / P_X:+.4f} with z fixed, or dz/dy = {-T / P_Z:+.4f} with x fixed")

## 6. Both components, from the beam's x scans

The x scans across the beam on Nov 19-20 (Scan file entries 10584-10613, y = 4-50 mm) give where the jet crosses the beam at each y. The jet signal peaks in `aemexp2_ch3` and dips in `aemexp2_ch4` at the same x. A Gaussian fit to ch3 gives the centre. Scans are dropped if the peak lies outside the scanned range, its FWHM is outside 15-100 µm, or the ch3 and ch4 centres differ by more than 5 µm.

The beam is close to, but not exactly along, z. On Nov 18 the jet was centred at the same y for several z (y = 0 and y = 4). Its x centre moves by tan β per mm of z, where β is the beam's angle from z. The jet crosses the beam where x_c = const − (a − c tan β) y + tan β z, so the scans fix a − c tan β. The microscope fixes p_x a + p_z c, and the two together give the jet direction (a, c) = (dx/dh, dz/dh).

Keeping the jet crossing the same point of the beam needs dx/dy = −a and dz/dy = −c. The z settings used on Nov 19-20 (the Nov 17 line) are compared with that.

Errors are Monte Carlo draws of T, the scan slope, tan β, p_x and p_z. p_x and p_z come from single 0.1 mm steps, so they carry the ±10 µm position accuracy (±14 %); the second error leaves that term out. The microscope sees the first 2.5 mm of the jet and the scans see 4-50 mm, so a curved jet would bias the result.

In [ ]:
SCAN_DAY = "20261117"  # the Scan file is named after its first day; it holds Nov 17-20
RECENT = range(10584, 10614)  # Nov 19-20 x scans
Z_STEPS = [10562, 10563, 10565, 10570, 10582, 10583]  # Nov 18: same y, different z (y = 0 and y = 4)


def _peak(x, mu, sigma, height, b0, b1):
    return b0 + b1 * (x - mu) + height * np.exp(-0.5 * ((x - mu) / sigma) ** 2)


def xscan(f, entry):
    """Jet centre (mm) of one `ascan a_mp1_x` and the y, z it ran at; None if not such a scan or not clean."""
    g = f[f"entry{entry}"]
    m = g["measurement"]
    if not g["title"][()].decode().startswith("ascan a_mp1_x") or len(m["a_mp1_x"]) < 10:
        return None
    x, snap = m["a_mp1_x"][()], m["pre_scan_snapshot"]
    out = {"entry": entry, "y": snap["a_mp1_y"][()], "z": snap["a_mp1_z"][()]}
    for name, s in [("x_c", m["aemexp2_ch3"][()]), ("x_c_ch4", -m["aemexp2_ch4"][()])]:  # ch4 dips
        i = np.argmax(s)
        try:
            p = curve_fit(_peak, x, s, p0=[x[i], np.ptp(x) / 10, np.ptp(s), np.median(s), 0], maxfev=20000)[0]
        except RuntimeError:
            return None
        out[name], out[name + "_fwhm"] = p[0], 2.355 * abs(p[1])
    clean = (x.min() < out["x_c"] < x.max() and 0.015 < out["x_c_fwhm"] < 0.1
             and abs(out["x_c"] - out["x_c_ch4"]) < 0.005)
    return out if clean else None


with daq.open_h5(daq.scan_file(SCAN_DAY)) as f:
    rows = [r for e in [*RECENT, *Z_STEPS] if (r := xscan(f, e))]
xs = pd.DataFrame(rows).set_index("entry")
recent, zsteps = xs[xs.index.isin(RECENT)], xs[xs.index.isin(Z_STEPS)]
print(f"{len(recent)} of {len(RECENT)} recent scans kept; dropped: {sorted(set(RECENT) - set(recent.index))}")

# beam angle from z: slope of x_c against z within each fixed-y group
grp = zsteps.y.round()
dz = zsteps.z - zsteps.groupby(grp).z.transform("mean")
dxc = zsteps.x_c - zsteps.groupby(grp).x_c.transform("mean")
TAN_B = (dz * dxc).sum() / (dz**2).sum()
TAN_B_ERR = np.sqrt(((dxc - TAN_B * dz) ** 2).sum() / (len(dz) - grp.nunique() - 1) / (dz**2).sum())

# jet centre against y, with the beam-angle part of the z changes taken out
(G_Y, G0), cov = np.polyfit(recent.y, recent.x_c - TAN_B * recent.z, 1, cov=True)
G_Y_ERR = np.sqrt(cov[0, 0])
wander = 1e3 * np.std(recent.x_c - TAN_B * recent.z - np.polyval([G_Y, G0], recent.y))
DZ_SET = np.polyfit(recent.y, recent.z, 1)[0]  # z settings used
print(f"beam: tan b = {TAN_B:+.4f} ± {TAN_B_ERR:.4f} ({np.degrees(np.arctan(TAN_B)):+.1f} deg from z)")
print(f"scans: a - c tan b = {-G_Y:+.4f} ± {G_Y_ERR:.4f}, jet centre scatter {wander:.0f} µm rms")

# solve  a - c tan b = -G_Y  and  p_x a + p_z c = T,  with Monte Carlo errors
rng = np.random.default_rng(0)
N = 20000
STEP_REL = np.sqrt(2) * POS_ERR / 0.1  # relative error of p_x, p_z from one 0.1 mm step


def solve(step_rel):
    def draw(v, e):
        return v + e * rng.standard_normal(N)

    px, pz = draw(P_X, step_rel * abs(P_X)), draw(P_Z, step_rel * abs(P_Z))
    tb, gy, t = draw(TAN_B, TAN_B_ERR), draw(G_Y, G_Y_ERR), draw(T, T_ERR)
    c = (t + px * gy) / (pz + px * tb)
    return -gy + c * tb, c


(a_all, c_all), (a_pos, c_pos) = solve(STEP_REL), solve(0)
C_JET = (T + P_X * G_Y) / (P_Z + P_X * TAN_B)
A_JET = -G_Y + C_JET * TAN_B
print(f"jet: dx/dh = {A_JET:+.4f} ± {a_all.std():.4f} (± {a_pos.std():.4f}), "
      f"dz/dh = {C_JET:+.4f} ± {c_all.std():.4f} (± {c_pos.std():.4f}); "
      f"{np.degrees(np.arctan(np.hypot(A_JET, C_JET))):.1f} deg from the y axis")
print(f"to keep the same crossing point: dx/dy = {-A_JET:+.4f}, dz/dy = {-C_JET:+.4f}; "
      f"z settings used: dz/dy = {DZ_SET:+.4f}")
print(f"with those settings the crossing point moves {DZ_SET + C_JET:+.4f} mm along the beam per mm of y, "
      f"{(DZ_SET + C_JET) * (recent.y.max() - recent.y.min()):+.2f} mm from y = {recent.y.min():.0f} to {recent.y.max():.0f}")

In [ ]:
fig, (a, b) = plt.subplots(1, 2, figsize=(12, 5))
yy = np.linspace(0, 52, 2)
a.plot(recent.y, recent.x_c - TAN_B * recent.z, "o", ms=4)
a.plot(yy, np.polyval([G_Y, G0], yy), "-", lw=0.8, label=f"slope {G_Y:+.4f}")
a.set(xlabel="y (mm)", ylabel="jet centre x - tan b z (mm)", title="Nov 19-20 x scans")
a.legend()

# (dx/dy, dz/dy) plane: the microscope line, the beam-scan line, and where they cross
gx = np.linspace(-0.12, 0.08, 2)
gz = np.linspace(-0.14, 0.06, 2)
b.fill_between(gx, (-T - T_ERR - P_X * gx) / P_Z, (-T + T_ERR - P_X * gx) / P_Z, alpha=0.3)
b.plot(gx, (-T - P_X * gx) / P_Z, label="microscope: jet stays in place in the picture")
b.plot(G_Y + TAN_B * gz, gz, label="x scans: jet stays on the beam")
b.errorbar(-A_JET, -C_JET, xerr=a_all.std(), yerr=c_all.std(), fmt="ko", ms=4, label="both: same crossing point")
b.plot(G_Y + TAN_B * DZ_SET, DZ_SET, "C3s", label="settings used Nov 19-20")
b.set(xlabel="dx/dy (mm/mm)", ylabel="dz/dy (mm/mm)", aspect="equal")
b.axhline(0, color="k", lw=0.5)
b.axvline(0, color="k", lw=0.5)
b.xaxis.set_major_locator(plt.MaxNLocator(5))
b.grid(alpha=0.3)
b.legend(fontsize=7)
fig.tight_layout()